In [1]:
import QuantLib as ql
import pandas as pd

In [2]:
today = ql.Date(28, ql.July, 2025)
ql.Settings.instance().evaluationDate = today

<h2>A complicated family</h2>

Read at <a href="https://www.quantlibguide.com/Cash-flow%20analysis.html">Site</a> 

<h2>Lost in translation</h2>

This poses a problem. Let's take a sample fixed-rate bond as an example.

In [3]:
schedule = ql.MakeSchedule(
    effectiveDate = ql.Date(8, ql.April, 2025),
    terminationDate = ql.Date(8, ql.April, 2030),
    frequency = ql.Semiannual,
    calendar = ql.TARGET(),
    convention= ql.Following,
    backwards=True
)

settlement_days = 3
face_amount = 10_000
coupon_rates= [0.03]

bond = ql.FixedRateBond(
    settlementDays = 3,
    faceAmount = 10_000,
    schedule=schedule,
    coupons =[0.03],
    paymentDayCounter=ql.Thirty360(ql.Thirty360.BondBasis)
)

We can extract its cash flows and use the CashFlow interface to display their dates and amounts:

In [4]:
cashflows = bond.cashflows()

data = []
for cf in cashflows:
    data.append((cf.date(),cf.amount()))

pd.DataFrame(data, columns=["date","amount"]).style.format(
    {"amount":"{:.2f}"}
)

,date,amount
0,"October 8th, 2025",150.00
1,"April 8th, 2026",150.00
2,"October 8th, 2026",150.00
3,"April 8th, 2027",150.00
4,"October 8th, 2027",150.00
5,"April 10th, 2028",151.67
6,"October 9th, 2028",149.17
7,"April 9th, 2029",150.00
8,"October 8th, 2029",149.17
9,"April 8th, 2030",150.00


<h2>Cash-flow analysis, at last</h2>

Given the functions above, we can collect a lot more information when cycling over cash flows. For instance, here we detect the coupons by trying to cast them and use their specialized interface to extract nominal, rate and accrual period; when the cast fail (that is, for the redemption) we fall back to extracting date and amount. By selecting the correct casting function, we can analyze cash flows from other kinds of bonds and collect the relevant information in each case.

In [6]:
data = []
for cf in cashflows:
    c= ql.as_coupon(cf)
    if c is not None:
        data.append(
            (
                c.date(),
                c.nominal(),
                c.rate(),
                c.accrualPeriod(),
                c.amount()
            )
        )
    else:
        data.append((cf.date(),None,None,None,cf.amount()))

pd.DataFrame(
    data, columns=["date", "nominal","rate","accrual_period","amount"]
).style.format(
    {
        "nominal":"{:.0f}",
        "rate":"{:.1%}",
        "accrual period":"{:.4f}",
        "amount":"{:.2f}"
    }
)

,date,nominal,rate,accrual_period,amount
0,"October 8th, 2025",10000,3.0%,0.500000,150.00
1,"April 8th, 2026",10000,3.0%,0.500000,150.00
2,"October 8th, 2026",10000,3.0%,0.500000,150.00
3,"April 8th, 2027",10000,3.0%,0.500000,150.00
4,"October 8th, 2027",10000,3.0%,0.500000,150.00
5,"April 10th, 2028",10000,3.0%,0.505556,151.67
6,"October 9th, 2028",10000,3.0%,0.497222,149.17
7,"April 9th, 2029",10000,3.0%,0.500000,150.00
8,"October 8th, 2029",10000,3.0%,0.497222,149.17
9,"April 8th, 2030",10000,3.0%,0.500000,150.00


<h2>Other functions</h2>

QuantLib provides a few other functions that act on a sequence of cash flows, rather than a single one. They are grouped as static methods of the CashFlows class; for instance, they include functions to calculate the present value and basis-point sensitivity, given a discount curve.

In [7]:
discount_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(0, ql.TARGET(), 0.04, ql.Actual360())
)

In [8]:
ql.CashFlows.npv(cashflows, discount_curve, False)

9625.543550654686

In [9]:
ql.CashFlows.bps(cashflows, discount_curve, False)

4.535150508988854

The last False parameter in the two calls above specifies that, if one of the cash flows were paid on the evaluation date, it should not be included.

Of course, these functions can also be used on a single cash flow by passing them a list with a single element; below, they are used to augment the analysis we performed earlier.

In [11]:
def npv(c):
    return ql.CashFlows.npv([c], discount_curve, False)

def bps(c):
    return ql.CashFlows.bps([c], discount_curve, False)

data = []
for cf in cashflows:
    c= ql.as_coupon(cf)
    if c is not None:
        data.append(
            (
                c.date(),
                c.nominal(),
                c.rate(),
                c.accrualPeriod(),
                c.amount(),
                npv(c),
                bps(c)
            )
        )
    else:
        data.append(
            (cf.date, None, None, None, cf.amount(), npv(cf), None)
        )

pd.DataFrame(
    data,
    columns = [
        "date",
        "nominal",
        "rate",
        "accrual period",
        "amount",
        "NPV",
        "BPS"
    ]
).style.format(
    {
        "nominal":"{:.0f}",
        "rate":"{:.1%}",
        "accrual period":"{:.4f}",
        "NPV":"{:.2f}",
        "BPS":"{:.2f}"
    }
)

,date,nominal,rate,accrual period,amount,NPV,BPS
0,"October 8th, 2025",10000,3.0%,0.5000,150.000000,148.80,0.50
1,"April 8th, 2026",10000,3.0%,0.5000,150.000000,145.83,0.49
2,"October 8th, 2026",10000,3.0%,0.5000,150.000000,142.89,0.48
3,"April 8th, 2027",10000,3.0%,0.5000,150.000000,140.03,0.47
4,"October 8th, 2027",10000,3.0%,0.5000,150.000000,137.21,0.46
5,"April 10th, 2028",10000,3.0%,0.5056,151.666667,135.91,0.45
6,"October 9th, 2028",10000,3.0%,0.4972,149.166667,131.00,0.44
7,"April 9th, 2029",10000,3.0%,0.5000,150.000000,129.09,0.43
8,"October 8th, 2029",10000,3.0%,0.4972,149.166667,125.80,0.42
9,"April 8th, 2030",10000,3.0%,0.5000,150.000000,123.97,0.41
